# SQL 基礎查詢與 SQLAlchemy ORM 建表邏輯

- 目標：掌握關聯式資料庫與 SQL 基礎語法，並精通 **Python SQLAlchemy ORM 的實體模型設計**。學會如何將半導體產線經典的**批號/晶圓/晶粒（Lot / Wafer / Die）三層追蹤（Traceability）架構 轉化為關聯式資料庫 Schema，並實作良率預測紀錄表 YieldPredictionRecord 的建表與操作邏輯**。


## 1. 半導體數據精髓：Lot / Wafer / Die 三層追蹤架構設計

- 在半導體製造與測試中，產品追蹤的層級具有嚴格的父子關聯：
    -   1. **批號 (Lot)**：最上層，一個 Lot 通常包含多達 25 片晶圓。
    -   2. **晶圓 (Wafer)**：中層，每片晶圓有其獨特的序號（如 Wafer No 1~25）。
    -   3. **粒 (Die)**：底層，晶圓上的每個晶片，由 X, Y 座標唯一決定，是最終測試與封裝的基本單位。
- 利用**資料庫的外鍵（Foreign Key）與級聯刪除（Cascade Delete）機制**，可以完美維持這三層資料的完整性（Referential Integrity）。


## 2. SQLAlchemy ORM 模型與 YieldPredictionRecord 建表實作

- 實戰場景：我們將使用 **SQLite（記憶體模式）來模擬資料庫環境**。我們需要為專案中的良率預測模組設計一張 YieldPredictionRecord 資料表，用來記錄每次模型跑出來的預測良率、實際良率、機台編號以及警報狀態（Alert Status）。


In [ ]:
from datetime import datetime
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    DateTime,
    ForeignKey,
)
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# 1. 初始化資料庫引擎（使用記憶體 SQLite 進行實作測試）
engine = create_engine("sqlite:///:memory:", echo=False)
Base = declarative_base()

# ==========================================
# 📊 設計符合 Lot/Wafer/Die 階層的實體模型
# ==========================================


class LotTable(Base):
    __tablename__ = "lots"
    id = Column(Integer, primary_key=True, autoincrement=True)
    lot_id = Column(
        String(50), unique=True, nullable=False, index=True
    )  # 建立索引提升查詢速度
    product_code = Column(String(50), nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)

    # 建立與 Wafer 的一對多關聯
    wafers = relationship(
        "WaferTable", back_populates="lot", cascade="all, delete-orphan"
    )


class WaferTable(Base):
    __tablename__ = "wafers"
    id = Column(Integer, primary_key=True, autoincrement=True)
    lot_id = Column(
        String(50), ForeignKey("lots.lot_id", ondelete="CASCADE"), nullable=False
    )
    wafer_no = Column(Integer, nullable=False)  # 晶圓片號 (1~25)

    lot = relationship("LotTable", back_populates="wafers")
    # 建立與良率預測紀錄的一對多關聯
    prediction_records = relationship("YieldPredictionRecord", back_populates="wafer")


class YieldPredictionRecord(Base):
    """對應專案：YieldPredictionRecord 建表邏輯"""

    __tablename__ = "yield_prediction_records"

    id = Column(Integer, primary_key=True, autoincrement=True)
    wafer_db_id = Column(
        Integer, ForeignKey("wafers.id", ondelete="CASCADE"), nullable=False
    )
    tool_id = Column(String(50), nullable=False)  # 測試機台 ID
    predicted_yield = Column(Float, nullable=False)  # 模型預測良率
    actual_yield = Column(Float, nullable=True)  # 實際良率（後續回填）
    alert_status = Column(
        String(20), default="NORMAL"
    )  # 警報狀態 (NORMAL, WARNING, CRITICAL)
    prediction_time = Column(DateTime, default=datetime.utcnow)  # 預測時間戳記

    wafer = relationship("WaferTable", back_populates="prediction_records")


# 2. 在資料庫中實體化建立所有定義好的資料表
Base.metadata.create_all(engine)
print("✅ 資料表 'lots', 'wafers', 'yield_prediction_records' 已成功建立！")


## 3. 資料庫 CRUD 與高階 SQL 聯結查詢（JOIN）

- 資料表建好後，我們透過 Session 來新增（Create）、讀取（Read）、並實作 SQL 基礎的聯結查詢（JOIN）與群組計算（GROUP BY）。


In [ ]:
# 1. 建立 Session
Session = sessionmaker(bind=engine)
session = Session()

# 2. 資料寫入 (Create)：新增一個批次、兩片晶圓，並塞入良率預測數據
print("\n>>> 正在寫入模擬晶圓測試數據...")
new_lot = LotTable(lot_id="LOT_A2026", product_code="RF_CHIP_X")
session.add(new_lot)
session.commit()  # 必須先 commit 讓外鍵可用

wafer1 = WaferTable(lot_id="LOT_A2026", wafer_no=1)
wafer2 = WaferTable(lot_id="LOT_A2026", wafer_no=2)
session.add_all([wafer1, wafer2])
session.commit()

# 為兩片晶圓寫入機器學習模型的預測紀錄
record1 = YieldPredictionRecord(
    wafer_db_id=wafer1.id,
    tool_id="TOOL_01",
    predicted_yield=98.5,
    actual_yield=98.2,
    alert_status="NORMAL",
)
record2 = YieldPredictionRecord(
    wafer_db_id=wafer2.id,
    tool_id="TOOL_01",
    predicted_yield=91.2,
    actual_yield=89.5,
    alert_status="CRITICAL",
)
session.add_all([record1, record2])
session.commit()

# 3. 高階查詢 (Read & JOIN & GROUP BY)：
# 等同於原生 SQL:
# SELECT l.lot_id, w.wafer_no, r.predicted_yield, r.alert_status
# FROM yield_prediction_records r
# JOIN wafers w ON r.wafer_db_id = w.id
# JOIN lots l ON w.lot_id = l.lot_id;
print("\n🔍 執行多表聯結查詢 (SQL JOIN) 結果：")
query_results = (
    session.query(
        LotTable.lot_id,
        WaferTable.wafer_no,
        YieldPredictionRecord.predicted_yield,
        YieldPredictionRecord.alert_status,
    )
    .join(WaferTable, LotTable.lot_id == WaferTable.lot_id)
    .join(YieldPredictionRecord, WaferTable.id == YieldPredictionRecord.wafer_db_id)
    .all()
)

for row in query_results:
    print(
        f"📦 批號: {row.lot_id} | 晶圓號: #{row.wafer_no} | 預測良率: {row.predicted_yield}% | 狀態: {row.alert_status}"
    )

# 關閉會話
session.close()


- 總結：在我們開發半導體測試或自動化分析系統時，良好的資料模型設計是底座。我非常熟悉關聯式資料庫的 Lot / Wafer / Die 三層追蹤架構（Traceability Schema）。在資料表設計上，我會對批號（Lot ID）與晶圓號建立 資料庫索引（Index）與外鍵（Foreign Key）連動，防止產線歷史資料孤立。在專案的實作中，我利用 SQLAlchemy ORM 模型 建立了 YieldPredictionRecord 表。這樣做的好處是能將機器學習模型的預測數據、實際良率回填以及警報狀態（alert_status），直接抽象化為 Python 物件進行安全操作。在需要巨量分析時，我也能流暢地撰寫多表聯結（JOIN）與 GROUP BY 的語法，快速拉出特定機台（Tool）在特定批次下的良率表現，確保資料架構兼具工程優雅性與產線擴充性。
